# STAR testing - Hamilton STAR / STARLet

Hardware validation notebook for the v1 STAR driver (`pylabrobot.hamilton.star`).

**This driver is currently read-only.** It emits exactly five firmware commands - `RM`, `QM`,
`RU`, `UA` and `RT` - all of which are configuration or sensor reads. It cannot initialise a
module, home an axis, or move anything. The legacy backend's `setup()` additionally ran
`pre_initialize_instrument`, `move_all_channels_in_z_safety`, `initialize_pip`, `initialize_iswap`,
`park_iswap` and `initialize_core_96_head`; none of those are ported yet.

That makes this notebook unusually safe to run - but connecting still takes the USB device, so do
not run it while another process owns the machine.

Ordered safe -> less safe. Cells that touch hardware are gated on `protocol_mode == "execution"`
and default to simulation.

## 1- Run identity

In [1]:
# --- Run identity ---
protocol_mode = "execution"  # simulation OR execution
user_name = "star_user"
run_identifier = "star_v1_validation"

# --- Device selection (only needed with more than one Hamilton on USB) ---
device_address = None  # USB address, e.g. 3
serial_number = None  # USB serial, e.g. "1234567"

## 2- Imports

In [2]:
from pylabrobot.hamilton.star.driver.master import STARDriver

## 3- Logging

Uses PyLabRobot's own `setup_logger`, exactly as every other PLR run does: a single
date-stamped file per day, appended to across runs. Both the file and the notebook are at
`IO` level, so every byte sent to and received from the machine is visible and recorded.

Re-running this cell is safe: `setup_logger` replaces the file handler and `verbose`
replaces the console handler, rather than stacking a second one of each.


In [3]:
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

log_dir = f"_logs/{protocol_mode}"

# PLR's own logger setup: one date-stamped file per day, appended to across runs. Re-running this
# cell replaces the file handler rather than stacking a second one, so lines are never duplicated.
pylabrobot.setup_logger(log_dir, level=LOG_LEVEL_IO)

# Console at IO level too: every byte sent and received appears in the notebook.
pylabrobot.verbose(True, level=LOG_LEVEL_IO)

logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)
print(f"appending to {log_dir}/pylabrobot-<YYYYMMDD>.log")

2026-08-14 15:57:48,600 - pylabrobot - INFO - --- star_v1_validation (execution) ---


appending to _logs/execution/pylabrobot-<YYYYMMDD>.log


## 4- Connect to the real device (read-only)

`setup()` opens the USB link, starts the reply router, and asks the machine what it is. It sends
no command that moves anything.

In [4]:
star = None

if protocol_mode == "execution":
  star = STARDriver(device_address=device_address, serial_number=serial_number)
  await star.setup()
  print("connected. channels:", star.num_channels)
else:
  print("simulation mode - skipped. set protocol_mode = 'execution' to connect.")

2026-08-14 15:57:48,611 - pylabrobot.io.usb - INFO - Finding USB device...
2026-08-14 15:57:48,629 - pylabrobot.io.usb - INFO - Found USB device.
2026-08-14 15:57:48,632 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-08-14 15:57:51,636 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RMid0001'
2026-08-14 15:57:51,703 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytea

connected. channels: 8


## 5- Device identity - what is this machine? (read-only)

In [5]:
if star is not None:
  c = star.configuration
  print(f"PIP channels          : {c.num_pip_channels}")
  print(f"PIP type              : {'1000uL' if c.pip_type_1000ul else '300uL'}")
  print(f"autoload installed    : {c.autoload_installed}")
  print(f"wash stations         : 1={c.wash_station_1_installed}  2={c.wash_station_2_installed}")
  print(f"deck size (slots)     : {c.instrument_size_slots}")
  print(f"tip waste x           : {c.tip_waste_x_position} mm")
  print(f"iSWAP gripper wide    : {c.iswap_gripper_wide}")

PIP channels          : 8
PIP type              : 1000uL
autoload installed    : True
wash stations         : 1=False  2=False
deck size (slots)     : 54
tip waste x           : 1340.0 mm
iSWAP gripper wide    : True


## 6- Arm geometry (read-only)

In [6]:
if star is not None:
  for side in ("left", "right"):
    arm = getattr(star.configuration, f"{side}_arm")
    if arm is None:
      print(f"{side:5s}: not installed")
      continue
    print(f"{side:5s}: {arm.model}")
    print(
      f"       width {arm.width} mm, travel {arm.x_range} mm, workspace {arm.workspace_range} mm"
    )
    print(f"       reference point: {arm.reference_point}")
    print(
      f"       modules: pip={arm.pip_installed} iswap={arm.iswap_installed} "
      f"head96={arm.head96_installed} xl={arm.xl_channels_installed}"
    )

left : hamilton_legacy_star_dual_rail_arm
       width 354.0 mm, travel (95.0, 1340.2) mm, workspace (-323.2, 1517.2) mm
       reference point: center
       modules: pip=True iswap=True head96=True xl=False
right: not installed


## 7- Sensor read: tip presence (read-only)

Each channel's sleeve sensor reports whether a tip is mounted. This reads sensors; it does not
move a channel.

In [7]:
if star is not None:
  presence = await star.request_tip_presence()
  for channel, has_tip in enumerate(presence):
    print(f"  channel {channel}: {'tip' if has_tip else '-'}")

2026-08-14 15:57:51,928 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RTid0006'
2026-08-14 15:57:51,970 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RTid0006er00/00rt0 0 0 0 0 0 0 0')


  channel 0: -
  channel 1: -
  channel 2: -
  channel 3: -
  channel 4: -
  channel 5: -
  channel 6: -
  channel 7: -


## 8- Raw command escape hatch

Anything not yet wrapped in a named method can be sent directly. **Only send commands you have
confirmed are read-only** - this bypasses every guard in the driver.

In [8]:
if star is not None:
  # C0 RF - request the master's firmware version. Read-only.
  print(await star.send_command(module="C0", command="RF"))

  # the same thing as a raw string, id included
  # print(await star.send_raw_command("C0RFid9999"))

2026-08-14 15:57:51,980 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RFid0007'
2026-08-14 15:57:51,996 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RFid0007er00/00rf7.6S 25 2021_11_05 (GRU C0)')


C0RFid0007er00/00rf7.6S 25 2021_11_05 (GRU C0)


## 9- Not yet implemented

The driver addresses only the gateway (`C0`). Every module below it is reached over the same
reply router, so each is a module to add rather than new plumbing:

| Module | Node | Status |
|---|---|---|
| Pipetting channels | `P1`-`PG` | not ported |
| 96-head | `H0` | not ported |
| iSWAP | `R0` | not ported |
| Autoload | `I0` | not ported |
| X-drives | `X0` | reported by `C0`, not addressed directly |
| Wash stations, pumps | `W1`/`W2`, `HW`/`HU`/`HV` | not ported |

Until those land, use the legacy backend for anything that has to move.

## 10- Teardown

In [9]:
if star is not None:
  await star.stop()
  print("disconnected.")

# The log is append-only and stays open for the rest of the session - nothing to close.

2026-08-14 15:57:52,008 - pylabrobot.io.usb - WARNING - Closing connection to USB device.


disconnected.
